# Normalise logRR curves to TMREL

Using method 3 from `Test_normalisation_method.ipynb` as recommended by Matthew Seymour

In [2]:
import os
import json
import numpy as np
import xarray as xr
import pathlib
import config
from utils.utils import require_dir

In [10]:
# === Setup ===
DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23")
TMREL_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23" / "TMREL")

# Load look up file
lookup_file = "country_location_ids.json"  # {"Armenia": 33, "Azerbaijan": 34, ...}
lookup_path = os.path.join(DIR, lookup_file)
with open(lookup_path, "r", encoding="utf-8") as f:
    country_to_id = json.load(f)

In [4]:
health_vars = ["CHRONIC_KIDNEY_DISEASE", "CARDIOMYOPATHY_MYOCARDITIS",
               "HYPERTENSIVE_HEART_DISEASE", "ISCHEMIC_HEART_DISEASE",
               "STROKE", "DIABETES", "INJ_ANIMAL", "INJ_DISASTER",
               "DROWNING", "HOMICIDE", "MECHANICAL_INJ",
               "OTHER_UNINTENTIONAL_INJ", "SUICIDE", "OTHER_TRANSPORT_INJ",
               "ROAD_TRANSPORT_INJ", "LOWER_RESPIRATORY_INFECTIONS", "COPD"]

In [5]:
def resample_tmrel(tmrel, n_logRR_draws):
    n_tmrel_draws = tmrel.sizes["draws"]
    # Resample TMREL draws (with replacement) to match logRR's draw count
    rng = np.random.default_rng(42)  # fixed seed: reproducible
    resample_idx = rng.integers(0, n_tmrel_draws, size=n_logRR_draws)

    tmrel_new_draws = tmrel.isel(draws=resample_idx)
    # re-label draws 0..999 so it aligns positionally with logRR's draws dim
    tmrel_new_draws = tmrel_new_draws.assign_coords(draws=np.arange(n_logRR_draws))
    return tmrel_new_draws

In [24]:
def normalize_to_tmrel(logrr: xr.DataArray, tmrel: xr.DataArray):
    """
    normalize with all tmrel draws to include tmrel uncertainty
    """
    ref_logRR = logrr.sel(daily_temperature=tmrel, method="nearest")  # dims: annual_temperature, draws
    logrr_normalized = logrr - ref_logRR  # broadcasts over daily_temperature
    return logrr_normalized

In [36]:
for health_VAR in health_vars:
    print(f"Processing {health_VAR}")

    # Load logRR curve for health variable
    RR_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23" / "RR_curves" / "temperature" / health_VAR)
    logRR_file = f"{health_VAR}_logRR_curve_samples.nc"
    logRR_path = os.path.join(RR_DIR, logRR_file)
    logRR = xr.open_dataarray(logRR_path)

    for name, cid in country_to_id.items():
        print(f"Processing country ID {cid}, {name}")

        # Load mean TMREL for country
        tmrel_file = f"tmrel_{cid}.nc"
        tmrel_path = os.path.join(TMREL_DIR, tmrel_file)
        tmrel_da = xr.open_dataarray(tmrel_path)

        n_logRR_draws = logRR.sizes["draws"]
        tmrel_extra_draws = resample_tmrel(tmrel_da, n_logRR_draws)

        logrr_normalized = normalize_to_tmrel(logRR, tmrel_extra_draws)

        description = (f"Normalised logRR curve by {name} tmrel, scripts by A.F. Wells 2026")
        logrr_normalized.attrs["description"] = description
        logrr_normalized.attrs["country name"] = name
        logrr_normalized.attrs["country ID"] = cid
        logrr_normalized.attrs["health_var"] = health_VAR

        out_file = f"{health_VAR}_norm_logRR_{cid}.nc"
        out_path = os.path.join(RR_DIR, out_file)
        print(f"Saving to {out_path}")
        logrr_normalized.to_netcdf(out_path)

print("All processing complete.")

Processing CHRONIC_KIDNEY_DISEASE
Processing country ID 33, Armenia
Saving to /glade/work/awells/workflow/GBD23/RR_curves/temperature/CHRONIC_KIDNEY_DISEASE/CHRONIC_KIDNEY_DISEASE_norm_logRR_33.nc
Processing country ID 34, Azerbaijan
Saving to /glade/work/awells/workflow/GBD23/RR_curves/temperature/CHRONIC_KIDNEY_DISEASE/CHRONIC_KIDNEY_DISEASE_norm_logRR_34.nc
Processing country ID 35, Georgia
Saving to /glade/work/awells/workflow/GBD23/RR_curves/temperature/CHRONIC_KIDNEY_DISEASE/CHRONIC_KIDNEY_DISEASE_norm_logRR_35.nc
Processing country ID 36, Kazakhstan
Saving to /glade/work/awells/workflow/GBD23/RR_curves/temperature/CHRONIC_KIDNEY_DISEASE/CHRONIC_KIDNEY_DISEASE_norm_logRR_36.nc
Processing country ID 37, Kyrgyzstan
Saving to /glade/work/awells/workflow/GBD23/RR_curves/temperature/CHRONIC_KIDNEY_DISEASE/CHRONIC_KIDNEY_DISEASE_norm_logRR_37.nc
Processing country ID 38, Mongolia
Saving to /glade/work/awells/workflow/GBD23/RR_curves/temperature/CHRONIC_KIDNEY_DISEASE/CHRONIC_KIDNEY_DIS